# 70 Customer segmentation

Two complementary views of the customer base:

1. **Behavioural clusters**: k-means (k = 8) on 14 per-customer features (`avangrid.segmentation.FEATURES`): size, daily
   shape, heating and cooling response, seasonality, grid-peak coincidence, load factor, variability and solar. Every
   customer is in exactly one cluster. Customers with too little use for peer-relative analysis (average under 0.05
   kWh/hour) form a ninth group, "Minimal use / vacant".
2. **Program segments**: flags (a customer can have several) built from the appliance inventory (notebook 60) and
   the technology scores (notebook 50):

| Segment | Definition |
|---|---|
| `high_electrification` | EV (listed, candidate list or Level 2 charging), primary electric heating, or heat pump (listed or "heat pump likely") |
| `peak_contributor` | Top 10% by kWh used during the 100 highest grid hours |
| `seasonal` | 2+ months averaging under 30% of the customer's yearly average |
| `flexible_load` | 2+ controllable loads among central AC, EV charging, electric water heater, pool, hot tub |
| `hp_retrofit_candidate` | Central AC and no electric heating (likely fuel heating with ductwork: a heat pump can replace both) |
| `ev_adoption_candidate` | Solar home (PV list or PV model) with no EV signal: early-adopter households |
| `solar` | PV list or PV model score ≥ 0.5 |

**Choosing k.** k is chosen for **stability across random starts** (adjusted Rand index between runs) and for
interpretability. Silhouette is low (about 0.14) at every k, as usual for household load data, which varies
continuously rather than in crisp groups. The scree plot in section 4 adds variance accounted for, on training and
held-out customers.

Outputs: DuckDB table `customer_segments`; in `onedrive-export/Data/eda/`: `customer_segments.csv`,
`segment_summary.csv`, `segment_profiles.png`, `segment_peak_share.png`, `segmentation_scree.png`.

Run after notebook 60. It takes about a minute.

## Setup

In [ ]:
import os
os.environ['OMP_NUM_THREADS'] = '1'  # must be set before importing scikit-learn

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import polars as pl
from sklearn.cluster import KMeans
from sklearn.model_selection import KFold

import avangrid  # noqa: F401
from avangrid import appliances, cache, config, db, features, grids, segmentation as seg, signatures

## 1. Behavioural clusters

Features are capped at their 1st and 99th percentiles and scaled by their median and interquartile range (or the
standard deviation, for mostly-zero features whose interquartile range is degenerate), then clustered with k-means.
`choose_k` records the stability, silhouette and Davies–Bouldin index for k = 4–12. Cluster names are assigned from
each cluster's median features in a fixed order, so they don't depend on k-means' arbitrary numbering.

In [ ]:
K_CLUSTERS = 8
hd = grids.load_hourly(verbose=False)
D, _ = features.load_daily_signals(hd, verbose=False)
F = appliances.appliance_features(hd.kwh, hd.temp_hourly, hd.dates, hd.weekend)
sig = signatures.load_signatures(hd, D, verbose=False)
X = seg.segmentation_features(hd.kwh, hd.dates, F, sig)
Z = seg.robust_standardize(X)
cache.save_results('segmentation_k_choice', seg.choose_k(Z))
labels = KMeans(K_CLUSTERS, n_init=20, random_state=0).fit_predict(Z)
names = seg.name_clusters(X, labels)
cluster_name = np.array([names[c] for c in labels])

## 2. Program segments

The minimal-use customers are added as the ninth group. Saved as the DuckDB table `customer_segments`.

In [ ]:
# ---------------------------------------------------------------- program segments
cust = pl.DataFrame({'CUSTOMER_NUMBER': hd.customer_numbers})
inv = cust.join(db.sql('SELECT * FROM appliance_inventory'), on='CUSTOMER_NUMBER', how='left')
tp = cust.join(db.sql('SELECT * FROM technology_predictions'), on='CUSTOMER_NUMBER', how='left')
cands = pl.read_csv(config.EDA_DIR / 'technology_candidates.csv')
in_cand = lambda t: np.isin(hd.customer_numbers, cands.filter(pl.col('technology') == t)['CUSTOMER_NUMBER'].to_numpy())

ev = tp['ev_listed'].to_numpy() | in_cand('EV') | (inv['ev_charging_type'] == 'Level 2 likely').to_numpy()
hp = tp['hp_listed'].to_numpy() | (inv['heating_type'] == 'heat pump likely').to_numpy()
solar = tp['pv_listed'].to_numpy() | (tp['pv_score'].fill_null(0).to_numpy() >= 0.5)
elec_heat_primary = inv['elec_heat_primary_flag'].to_numpy()
central_ac = (inv['ac_type'] == 'central likely').to_numpy()
ev_any = ev | (inv['ev_charging_type'] != 'none detected').to_numpy()
flex_count = (central_ac.astype(int) + ev_any + inv['water_heater_flag'].to_numpy() + inv['pool_flag'].to_numpy()
              + inv['hot_tub_flag'].to_numpy())
grid_peak_kwh = sig['grid_peak_kwh'].to_numpy()

segments = cust.with_columns(
    pl.Series('cluster', labels), pl.Series('segment', cluster_name),
    pl.Series('high_electrification', ev | elec_heat_primary | hp),
    pl.Series('peak_contributor', grid_peak_kwh >= np.nanpercentile(grid_peak_kwh, 90)),
    pl.Series('seasonal', X[:, seg.FEATURES.index('empty_months')] >= 2),
    pl.Series('flexible_load', flex_count >= 2),
    pl.Series('hp_retrofit_candidate', central_ac & ~inv['elec_heat_flag'].to_numpy()),
    pl.Series('ev_adoption_candidate', solar & ~ev_any),
    pl.Series('solar', solar),
    pl.Series('flexible_load_count', flex_count),
    pl.Series('grid_peak_kwh', grid_peak_kwh),
    *[pl.Series(f, X[:, i]) for i, f in enumerate(seg.FEATURES)],
)

# The near-zero customers (in ami_analysis, not in the peer groups / hourly arrays)
low = db.sql('''SELECT CUSTOMER_NUMBER FROM ami_analysis GROUP BY 1
                HAVING CUSTOMER_NUMBER NOT IN (SELECT CUSTOMER_NUMBER FROM customer_peer_groups)''')
low = low.with_columns(pl.lit(K_CLUSTERS).cast(segments['cluster'].dtype).alias('cluster'),
                       pl.lit('Minimal use / vacant').alias('segment'))
all_segments = pl.concat([segments, low], how='diagonal').sort('CUSTOMER_NUMBER')
flag_cols = ['high_electrification', 'peak_contributor', 'seasonal', 'flexible_load', 'hp_retrofit_candidate',
             'ev_adoption_candidate', 'solar']
all_segments = all_segments.with_columns([pl.col(c).fill_null(False) for c in flag_cols])
with db.connect(read_only=False) as con:
    con.register('seg_df', all_segments)
    con.execute('CREATE OR REPLACE TABLE customer_segments AS SELECT * FROM seg_df')
all_segments.write_csv(config.EDA_DIR / 'customer_segments.csv')
cache.save_results('customer_segments', all_segments)

## 3. How much of the energy and the grid's peaks each segment accounts for

For each cluster: its share of customers, of annual energy, and of use during the grid's 100 highest hours (and the 50
highest in winter and summer), plus typical size and load shape.

In [ ]:
# ---------------------------------------------------------------- summary
K = hd.kwh
system = np.zeros((K.shape[1], 24))
for s in range(0, K.shape[0], 3000):
    system += np.nansum(np.asarray(K[s:s + 3000], dtype=np.float64), 0)
flat = system.ravel()
month = np.repeat(hd.dates.dt.month().to_numpy(), 24)
top = lambda mask, k: np.flatnonzero(mask)[np.argsort(-flat[mask])[:k]]
peak_hours = {'all': top(np.ones_like(flat, bool), 100), 'winter': top(np.isin(month, [12, 1, 2]), 50),
              'summer': top(np.isin(month, [6, 7, 8]), 50)}
energy = np.zeros(K.shape[0]); peak_kwh = {k: np.zeros(K.shape[0]) for k in peak_hours}
for s in range(0, K.shape[0], 3000):
    kf = np.asarray(K[s:s + 3000], dtype=np.float64).reshape(-1, K.shape[1] * 24)
    energy[s:s + 3000] = np.nanmean(kf, 1) * kf.shape[1]
    for k, idx in peak_hours.items():
        peak_kwh[k][s:s + 3000] = np.nansum(kf[:, idx], 1)

summ = segments.with_columns(pl.Series('energy', energy), *[pl.Series(f'peak_{k}', v) for k, v in peak_kwh.items()],
                             pl.Series('central_ac', central_ac), pl.Series('ev', ev_any), pl.Series('elec_heat', inv['elec_heat_flag'].to_numpy()))
tot = {c: float(summ[c].sum()) for c in ['energy', 'peak_all', 'peak_winter', 'peak_summer']}
summary = (summ.group_by('segment').agg(
    pl.len().alias('customers'),
    (pl.len() / summ.height).alias('share_customers'),
    (pl.col('energy').sum() / tot['energy']).alias('share_energy'),
    (pl.col('peak_all').sum() / tot['peak_all']).alias('share_grid_peak'),
    (pl.col('peak_winter').sum() / tot['peak_winter']).alias('share_winter_peak'),
    (pl.col('peak_summer').sum() / tot['peak_summer']).alias('share_summer_peak'),
    (pl.col('log_mean_kwh').exp().median()).alias('median_kw'),
    pl.col('load_factor').median(), pl.col('grid_peak_ratio').median(),
    pl.col('central_ac').mean().alias('central_ac'), pl.col('elec_heat').mean(), pl.col('ev').mean().alias('ev_signal'),
    pl.col('solar').mean(), *[pl.col(c).mean() for c in ['high_electrification', 'peak_contributor', 'flexible_load',
                                                         'hp_retrofit_candidate']],
).sort('share_grid_peak', descending=True))
summary.write_csv(config.EDA_DIR / 'segment_summary.csv')
cache.save_results('segment_summary', summary)
overlays = pl.DataFrame([{'segment': c, 'customers': int(all_segments[c].sum()),
                          'share_customers': float(all_segments[c].mean())} for c in flag_cols])
cache.save_results('segment_overlays', overlays)
with pl.Config(tbl_rows=-1, tbl_cols=-1, tbl_width_chars=250, tbl_hide_dataframe_shape=True, float_precision=3, fmt_str_lengths=35):
    print(summary)
    print(overlays)
    print('peak hours (top 100):', sorted({(int(i) // 24, int(i) % 24 + 1) for i in peak_hours['all']})[:5], '...')

### Charts: average day by segment and season, and share of customers vs share of the grid peak

In [ ]:
# ---------------------------------------------------------------- charts
order = summary['segment'].to_list()
COLORS = ['#0072B2', '#E69F00', '#009E73', '#CC79A7', '#D55E00', '#56B4E9', '#F0E442', '#8c8c8c']
color = {s: COLORS[i % len(COLORS)] for i, s in enumerate(order)}
month_d = hd.dates.dt.month().to_numpy()
seasons = {'Winter (Dec-Feb)': np.isin(month_d, [12, 1, 2]), 'Summer (Jun-Aug)': np.isin(month_d, [6, 7, 8])}
fig, axes = plt.subplots(2, 4, figsize=(16, 7.5), sharex=True)
for ax, s in zip(axes.flat, order):
    idx = np.flatnonzero(cluster_name == s)
    for (lab, days), style in zip(seasons.items(), ['-', '--']):
        prof = np.zeros(24)
        for j in range(0, len(idx), 3000):
            prof += np.nansum(np.nanmean(np.asarray(K[idx[j:j + 3000]])[:, days], 1), 0)
        ax.plot(range(1, 25), prof / len(idx), color=color[s], linestyle=style, linewidth=2.2, label=lab)
    r = summary.filter(pl.col('segment') == s).row(0, named=True)
    ax.set_title(f"{s}\n{r['customers']:,} homes · {r['share_grid_peak']:.0%} of grid peak", fontsize=9)
    ax.set_xticks(range(1, 25, 4))
    ax.set_ylim(bottom=0)
    ax.grid(color='#e5e5e5', linewidth=0.7)
    ax.spines[['top', 'right']].set_visible(False)
for ax in axes[1]:
    ax.set_xlabel('Hour ending')
for ax in axes[:, 0]:
    ax.set_ylabel('Average kWh per hour')
axes[0, 0].legend(frameon=False, fontsize=8)
fig.suptitle('Average daily profile of each customer segment, winter vs summer', y=1.0)
fig.tight_layout()
fig.savefig(config.EDA_DIR / 'segment_profiles.png', dpi=150)

fig, ax = plt.subplots(figsize=(10, 5))
y = np.arange(len(order))
ax.barh(y - 0.2, summary['share_customers'], height=0.38, color='#bbbbbb', label='Share of customers')
ax.barh(y + 0.2, summary['share_grid_peak'], height=0.38, color='#0072B2', label='Share of use in the 100 highest grid hours')
ax.set_yticks(y, order)
ax.invert_yaxis()
ax.xaxis.set_major_formatter(matplotlib.ticker.PercentFormatter(1.0))
for i, (a, b) in enumerate(zip(summary['share_customers'], summary['share_grid_peak'])):
    ax.text(b + 0.004, i + 0.2, f'{b:.0%}', va='center', fontsize=8)
ax.set_title('Which segments drive the grid peak?')
ax.grid(axis='x', color='#e5e5e5', linewidth=0.7)
ax.spines[['top', 'right']].set_visible(False)
ax.legend(frameon=False, loc='lower right')
fig.tight_layout()
fig.savefig(config.EDA_DIR / 'segment_peak_share.png', dpi=150)
print('saved charts')

## 4. Scree plot: variance accounted for, on training and held-out customers

**VAF** = 1 − within-cluster sum of squares ÷ total sum of squares: the share of the spread between customers that
the clusters explain, like R². It rises with k, so the "elbow" is where it stops rising quickly.

**Overfitting check (5-fold cross-validation).** For each fold, the standardization (percentile caps, median, spread)
and k-means are fitted on the other four folds only. The held-out customers are standardized with those training
values and assigned to the nearest training centroid; their VAF is computed around their own mean. If test VAF tracks
training VAF, the clusters describe the population rather than the particular customers they were fitted on.

The right panel repeats the stability from section 1, which is what k = 8 was chosen on.

The standardization is refitted on each training fold (the same rule as `avangrid.segmentation.robust_standardize`),
so held-out customers never influence it.

In [ ]:
K_CHOSEN = 8
KS = list(range(1, 16))
N_FOLDS = 5


def fit_standardizer(X):
    """Training-set values for avangrid.segmentation.robust_standardize, so test rows can reuse them."""
    lo, hi = np.nanpercentile(X, 1, axis=0), np.nanpercentile(X, 99, axis=0)
    Xc = np.clip(X, lo, hi)
    med = np.nanmedian(Xc, axis=0)
    spread = np.nanpercentile(Xc, 75, axis=0) - np.nanpercentile(Xc, 25, axis=0)
    spread = np.where(spread > 0.25 * np.nanstd(Xc, axis=0), spread, np.nanstd(Xc, axis=0) + 1e-9)
    return lo, hi, med, spread


def standardize(X, params):
    lo, hi, med, spread = params
    Z = (np.clip(X, lo, hi) - med) / spread
    return np.where(np.isnan(Z), 0.0, Z)


def vaf(Z, centers):
    """1 - within-cluster SS / total SS, with each row assigned to its nearest center."""
    d2 = ((Z[:, None, :] - centers[None, :, :]) ** 2).sum(2)
    within = d2.min(1).sum()
    total = ((Z - Z.mean(0)) ** 2).sum()
    return 1 - within / total

In [ ]:
# The minimal-use customers (cluster == K_CHOSEN) were added after clustering; leave them out.
s = db.sql('SELECT * FROM customer_segments ORDER BY CUSTOMER_NUMBER').filter(pl.col('cluster') < K_CHOSEN)
X = s.select(seg.FEATURES).to_numpy().astype(float)

rows = []
for fold, (tr, te) in enumerate(KFold(N_FOLDS, shuffle=True, random_state=0).split(X)):
    params = fit_standardizer(X[tr])
    Ztr, Zte = standardize(X[tr], params), standardize(X[te], params)
    for k in KS:
        km = KMeans(k, n_init=20, random_state=0).fit(Ztr)
        rows.append({'fold': fold, 'k': k, 'vaf_train': vaf(Ztr, km.cluster_centers_),
                     'vaf_test': vaf(Zte, km.cluster_centers_)})
folds = pl.DataFrame(rows)
res = (folds.group_by('k').agg(
    *[pl.col(c).mean().alias(c) for c in ['vaf_train', 'vaf_test']],
    *[pl.col(c).min().alias(f'{c}_min') for c in ['vaf_train', 'vaf_test']],
    *[pl.col(c).max().alias(f'{c}_max') for c in ['vaf_train', 'vaf_test']],
).sort('k').with_columns((pl.col('vaf_train') - pl.col('vaf_test')).alias('train_minus_test')))
cache.save_results('segmentation_vaf', res)
kc = cache.load_results('segmentation_k_choice')

In [ ]:
# Reference palette (light). Slots 1 and 2 of the categorical order; text and grid in neutral tokens.
TRAIN, TEST = '#2a78d6', '#eb6834'
TEXT, TEXT2, GRID, SURFACE = '#0b0b0b', '#52514e', '#e6e5e1', '#fcfcfb'
plt.rcParams.update({'font.size': 10, 'text.color': TEXT, 'axes.labelcolor': TEXT2, 'xtick.color': TEXT2,
                     'ytick.color': TEXT2, 'axes.edgecolor': GRID, 'figure.facecolor': SURFACE,
                     'axes.facecolor': SURFACE})
dot = dict(linewidth=2, marker='o', markersize=6, markeredgecolor=SURFACE, markeredgewidth=2, zorder=3)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.6))
ks = res['k'].to_numpy()
i8 = KS.index(K_CHOSEN)

# The two curves nearly coincide, so training is a wide solid line and held-out a dashed line with hollow markers
# on top: both stay visible where they overlap. (The fold-to-fold range is under 1 point, too thin to draw.)
ax1.plot(ks, res['vaf_train'], color=TRAIN, linewidth=4, marker='o', markersize=8, markerfacecolor=TRAIN,
         markeredgecolor=SURFACE, markeredgewidth=2, zorder=3, label='Training folds (mean of 5)')
ax1.plot(ks, res['vaf_test'], color=TEST, linewidth=1.6, linestyle=(0, (4, 3)), marker='o', markersize=5,
         markerfacecolor=SURFACE, markeredgecolor=TEST, markeredgewidth=1.6, zorder=4,
         label='Held-out fold (mean of 5)')
tr8, te8 = res['vaf_train'][i8], res['vaf_test'][i8]
ax1.plot(K_CHOSEN, tr8, marker='o', markersize=11, markerfacecolor='none', markeredgecolor=TEXT,
         markeredgewidth=1.5, zorder=4)
ax1.annotate(f'k = {K_CHOSEN} (chosen)\ntraining {tr8:.0%}, held-out {te8:.0%}', xy=(K_CHOSEN, tr8),
             xytext=(K_CHOSEN + 0.8, 0.28), fontsize=9, color=TEXT,
             arrowprops=dict(arrowstyle='-', color=TEXT2, linewidth=1))
ax1.set_title('Variance accounted for (VAF), 5-fold cross-validation', loc='left', fontsize=11, color=TEXT)
ax1.set_xlabel('Number of clusters (k)')
ax1.set_ylabel('VAF = 1 - within-cluster SS / total SS')
ax1.set_ylim(0, 0.7)
ax1.yaxis.set_major_formatter(matplotlib.ticker.PercentFormatter(1.0))
ax1.legend(frameon=False, loc='upper left', fontsize=9)

ax2.plot(kc['k'], kc['stability_ari'], color=TRAIN, markerfacecolor=TRAIN, **dot)
a8 = kc.filter(pl.col('k') == K_CHOSEN)['stability_ari'][0]
# First k above the chosen one where runs from different random starts stop agreeing.
k_unstable = int(kc.filter((pl.col('k') > K_CHOSEN) & (pl.col('stability_ari') < 0.9))['k'].min())
ax2.plot(K_CHOSEN, a8, marker='o', markersize=11, markerfacecolor='none', markeredgecolor=TEXT,
         markeredgewidth=1.5, zorder=4)
ax2.annotate(f'k = {K_CHOSEN}: {a8:.2f}', xy=(K_CHOSEN, a8), xytext=(K_CHOSEN - 3.2, 0.7),
             fontsize=9, color=TEXT, arrowprops=dict(arrowstyle='-', color=TEXT2, linewidth=1))
ax2.annotate(f'Unstable from k = {k_unstable}', xy=(k_unstable, kc.filter(pl.col('k') == k_unstable)['stability_ari'][0]),
             xytext=(k_unstable + 0.4, 0.5), fontsize=9, color=TEXT2,
             arrowprops=dict(arrowstyle='-', color=TEXT2, linewidth=1))
ax2.set_title('Stability across random starts (adjusted Rand index)', loc='left', fontsize=11, color=TEXT)
ax2.set_xlabel('Number of clusters (k)')
ax2.set_ylabel('Agreement between runs (1 = identical)')
ax2.set_ylim(0, 1.05)

for ax in (ax1, ax2):
    ax.set_xlim(0.5, 15.5)
    ax.set_xticks(KS)
    ax.grid(axis='y', color=GRID, linewidth=0.8)
    ax.set_axisbelow(True)
    ax.spines[['top', 'right', 'left']].set_visible(False)
    ax.tick_params(length=0)

gap = float(res['train_minus_test'].abs().max())
fig.suptitle(f'Choosing the number of customer segments ({s.height:,} customers, {len(seg.FEATURES)} features)',
             x=0.01, ha='left', fontsize=12, color=TEXT)
fig.text(0.01, 0.005, f'Held-out VAF stays within {gap * 100:.1f} percentage points of training VAF at every k '
         f'(no sign of overfitting). Stability collapses at k = {k_unstable}, so k = {K_CHOSEN} is the largest stable choice.',
         fontsize=8.5, color=TEXT2)
fig.tight_layout(rect=(0, 0.03, 1, 1))
out = config.EDA_DIR / 'segmentation_scree.png'
fig.savefig(out, dpi=200, facecolor=SURFACE)
with pl.Config(tbl_rows=-1, float_precision=4):
    print(res.select('k', 'vaf_train', 'vaf_test', 'train_minus_test'))
print(f'saved {out}')

## Tried but not used

| Tried | Why it isn't used |
|---|---|
| **Other numbers of clusters** (k = 4–12 compared; VAF up to k = 15) | Stability across random starts collapses above k = 8 (table below), and more clusters split groups that are hard to tell apart. Held-out VAF tracks training VAF at every k, so overfitting doesn't rule out any k; stability and interpretability decide. |
| **Silhouette and Davies–Bouldin** to choose k | Both are poor at every k (silhouette about 0.12–0.17), as usual for household load data that varies continuously. Computed and reported, but not used to choose. |
| **Scaling every feature by its interquartile range alone** | On the full data the mostly-zero solar feature `solar_midday_zero_share` got an interquartile range of 0.001, so it dominated k-means and put 89% of customers in one cluster. Features now fall back to the standard deviation when their interquartile range is under a quarter of it. |

In [ ]:
with pl.Config(tbl_hide_dataframe_shape=True, float_precision=3):
    print(cache.load_results('segmentation_k_choice'))